In [2]:
import os, shutil, gc, torch

for path in [
    "/kaggle/working/offload",
    "/kaggle/working/lora_adapter",
    "/kaggle/working/lora_adapter_v4_answer_only",
    "/kaggle/working/submission.zip",
    "/kaggle/working/ptxas-blackwell"
]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

gc.collect()
torch.cuda.empty_cache()
print("Cleaned old outputs.")

Cleaned old outputs.


In [3]:
import site

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import kagglehub
import mamba_ssm
import torch
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Dependencies working")

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


Dependencies working


In [4]:
import os, shutil, subprocess

SRC = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
DST = "/kaggle/working/ptxas-blackwell"

shutil.copy2(SRC, DST)
os.chmod(DST, 0o755)

os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = DST
os.environ["TRITON_PTXAS_PATH"] = DST

print("Copied ptxas-blackwell to:", DST)
print(subprocess.check_output([DST, "--version"]).decode())

Copied ptxas-blackwell to: /kaggle/working/ptxas-blackwell
ptxas-blackwell: NVIDIA (R) Ptx optimizing assembler
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Tue_May_27_02:18:05_PDT_2025
Cuda compilation tools, release 12.9, V12.9.86
Build cuda_12.9.r12.9/compiler.36037853_0



In [4]:
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")
print(torch.cuda.get_device_capability(0))

NVIDIA RTX PRO 6000 Blackwell Server Edition
94.97076416015625 GB
(12, 0)


In [5]:
import pandas as pd

TRAIN_PATH = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
TEST_PATH = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(train_df.shape)
print(test_df.shape)
train_df.head()

(9500, 3)
(3, 2)


,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",XXXVIII
4,001c63cb,"In Alice's Wonderland, secret encryption rules...",wizard creates secret


In [6]:
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)

OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16
)

print("Model loaded successfully.")

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded successfully.


In [7]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer ready")

Tokenizer ready


In [8]:
import torch
from torch.utils.data import Dataset, DataLoader

class AnswerOnlyReasoningDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=768):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        prompt = str(self.df.loc[idx, "prompt"])
        answer = str(self.df.loc[idx, "answer"])

        prefix = (
            "### Task\n"
            "Solve the reasoning puzzle and provide only the final answer.\n\n"
            "### Puzzle\n"
            f"{prompt}\n\n"
            "### Final Answer\n"
        )

        full_text = prefix + answer

        full_enc = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        prefix_enc = self.tokenizer(
            prefix,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_tensors="pt"
        )

        input_ids = full_enc["input_ids"].squeeze(0)
        attention_mask = full_enc["attention_mask"].squeeze(0)

        labels = input_ids.clone()
        labels[attention_mask == 0] = -100

        prefix_len = prefix_enc["input_ids"].shape[1]
        labels[:prefix_len] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

train_dataset = AnswerOnlyReasoningDataset(train_df, tokenizer, max_length=768)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0
)

print("Answer-only dataset ready:", len(train_dataset))

Answer-only dataset ready: 9500


In [9]:
print(f"Initializing LoRA adapter with rank={LORA_RANK}...")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Initializing LoRA adapter with rank=32...


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116


In [10]:
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for step, batch in enumerate(train_loader):
    batch = {k: v.to(model.device) for k, v in batch.items()}

    outputs = model(**batch)
    loss = outputs.loss

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print(f"Step {step+1} | Loss: {loss.item():.4f}")

    if step == 4:
        break

Step 1 | Loss: 1.0563
Step 2 | Loss: 3.4868
Step 3 | Loss: 0.5615
Step 4 | Loss: 0.9303
Step 5 | Loss: 4.6021


In [11]:
MAX_STEPS = 2000
GRAD_ACCUM_STEPS = 4

model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
optimizer.zero_grad()

for step, batch in enumerate(train_loader):
    batch = {k: v.to(model.device) for k, v in batch.items()}

    outputs = model(**batch)
    loss = outputs.loss / GRAD_ACCUM_STEPS
    loss.backward()

    if (step + 1) % GRAD_ACCUM_STEPS == 0:
        optimizer.step()
        optimizer.zero_grad()

    if (step + 1) % 10 == 0:
        print(f"Step {step+1} | Loss: {loss.item() * GRAD_ACCUM_STEPS:.4f}")

    if step + 1 >= MAX_STEPS:
        break

print("V4 answer-only 2000-step training complete.")

Step 10 | Loss: 1.9716
Step 20 | Loss: 0.0050
Step 30 | Loss: 0.0044
Step 40 | Loss: 0.8651
Step 50 | Loss: 1.4674
Step 60 | Loss: 0.0102
Step 70 | Loss: 0.7564
Step 80 | Loss: 1.9338
Step 90 | Loss: 1.2353
Step 100 | Loss: 2.5155
Step 110 | Loss: 0.0002
Step 120 | Loss: 0.8669
Step 130 | Loss: 0.4332
Step 140 | Loss: 0.6142
Step 150 | Loss: 0.8000
Step 160 | Loss: 5.2536
Step 170 | Loss: 1.9446
Step 180 | Loss: 0.2772
Step 190 | Loss: 3.1480
Step 200 | Loss: 2.5089
Step 210 | Loss: 1.3682
Step 220 | Loss: 1.7374
Step 230 | Loss: 0.0002
Step 240 | Loss: 0.5032
Step 250 | Loss: 3.5125
Step 260 | Loss: 0.7773
Step 270 | Loss: 2.9092
Step 280 | Loss: 0.2761
Step 290 | Loss: 1.5892
Step 300 | Loss: 1.1503
Step 310 | Loss: 0.0006
Step 320 | Loss: 3.0330
Step 330 | Loss: 0.0057
Step 340 | Loss: 0.6972
Step 350 | Loss: 1.1306
Step 360 | Loss: 0.0000
Step 370 | Loss: 0.7860
Step 380 | Loss: 0.1364
Step 390 | Loss: 2.4387
Step 400 | Loss: 3.1229
Step 410 | Loss: 3.1186
Step 420 | Loss: 2.6099
S

In [12]:
ADAPTER_DIR = "/kaggle/working/lora_adapter_v4_answer_only"

model.save_pretrained(ADAPTER_DIR)

print("Saved:", ADAPTER_DIR)

Saved: /kaggle/working/lora_adapter_v4_answer_only


In [13]:
import os, zipfile

ADAPTER_DIR = "/kaggle/working/lora_adapter_v4_answer_only"
ZIP_PATH = "/kaggle/working/submission.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(ADAPTER_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, ADAPTER_DIR)
            zipf.write(full_path, arcname)

print("Created:", ZIP_PATH)

Created: /kaggle/working/submission.zip


In [14]:
!ls -lh /kaggle/working/submission.zip
!unzip -l /kaggle/working/submission.zip | head -20

-rw-r--r-- 1 root root 3.0G Jun  9 17:53 /kaggle/working/submission.zip
Archive:  /kaggle/working/submission.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
     1045  2026-06-09 17:51   adapter_config.json
3522350336  2026-06-09 17:51   adapter_model.safetensors
     5312  2026-06-09 17:51   README.md
---------                     -------
3522356693                     3 files
